# Random forest benchmark: scikit-learn, Bankai, and LightGBM RF

This notebook compares binary-classification workloads on the same deterministic dataset (1,000 rows and 500 features by default):

1. **scikit-learn RF vs. Bankai:** both calculate permutation importance. The timer includes scikit-learn `fit` plus validation-set permutation importance; Bankai calculates OOB permutation importance inside `fit`.
2. **LightGBM RF vs. Bankai:** both use gain importance, and the timer covers `fit` only.

**Run in Google Colab:** choose a CPU runtime and select **Runtime → Run all**. The first code cell installs Bankai from PyPI plus the pinned scikit-learn and LightGBM versions. No Rust toolchain, repository clone, or manual setup is needed.

The scikit-learn and Bankai importance protocols both use accuracy decrease after feature permutation, but they are not identical: scikit-learn permutes a held-out validation set five times per feature, while Bankai computes out-of-bag importance across its trees. The first comparison therefore measures the complete fit-plus-permutation workload on both sides and labels the protocol difference.

Dataset generation, splitting, imports, estimator construction, and garbage collection are outside the timers. Each model gets one warm-up and three measured runs in alternating order; tables show median and range. `n_jobs=-1` requests all visible CPUs. The LightGBM comparison remains fit-only.

In [ ]:
%pip install -q "bankai-random-forest" "scikit-learn==1.9.1" "lightgbm==4.7.0"

## 1. Check the runtime and installed packages

The Bankai wheel on PyPI supports Colab's Linux x86-64 runtime and Python 3.11 or newer. This cell confirms the versions and available CPU count before starting the fits.

In [ ]:
import gc
import os
import platform
import statistics
import time

import lightgbm as lgb
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from joblib import cpu_count
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from importlib.metadata import version
from bankai_random_forest import BankaiRandomForestClassifier

print(f"Python: {platform.python_version()}")
print(f"Bankai: {version('bankai-random-forest')}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"LightGBM: {lgb.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Logical CPUs reported by OS: {os.cpu_count()}")
print(f"CPUs available to joblib: {cpu_count()}")

## 2. Generate and split one binary dataset

Both benchmarks use the same generated `X` and `y`. The notebook splits 1,000 rows into 800 training rows and 200 validation rows by default. All estimators fit on the training partition. The validation partition is used only for scikit-learn's permutation-importance calculation; Bankai calculates its permutation importance from out-of-bag rows.

In [ ]:
N_SAMPLES = 1_000
N_FEATURES = 500
N_RELEVANT = 90
DATA_SEED = 1789

rng = np.random.default_rng(DATA_SEED)
# The default 1,000 x 500 float32 matrix uses about 1.9 MiB.
X = rng.standard_normal((N_SAMPLES, N_FEATURES), dtype=np.float32)
coefficients = rng.choice(np.array([-1.0, 1.0], dtype=np.float32), size=N_RELEVANT)
coefficients *= rng.uniform(0.5, 1.5, size=N_RELEVANT).astype(np.float32)
signal = X[:, :N_RELEVANT] @ coefficients
signal /= np.sqrt(np.sum(coefficients ** 2))
probability = 1.0 / (1.0 + np.exp(-signal))
y = rng.binomial(1, probability).astype(np.uint8)

assert X.shape == (N_SAMPLES, N_FEATURES)
assert np.count_nonzero(coefficients) == N_RELEVANT
X_train, X_validation, y_train, y_validation = train_test_split(
    X, y, test_size=0.2, random_state=DATA_SEED, stratify=y
)
print(f"X: {X.shape}, dtype={X.dtype}, {X.nbytes / (1024**2):.1f} MiB")
print(f"train: {X_train.shape}; validation: {X_validation.shape}")
print(f"y class counts: {np.bincount(y).tolist()}")
print(f"Signal features: 0–{N_RELEVANT - 1}; noise features: {N_RELEVANT}–{N_FEATURES - 1}")

## 3. Shared benchmark helper

Estimator construction and garbage collection happen before timing. The helper always times `fit`; for the scikit-learn permutation comparison, a post-fit callback adds validation permutation importance to the same timed workload. The LightGBM section supplies no callback, so its timer remains fit-only.

In [ ]:
N_JOBS = -1
N_WARMUPS = 1
N_REPEATS = 3


def run_fit_benchmark(factories, baseline=None, post_fit=None):
    """Time fit and, when configured, the model's external importance calculation."""
    samples = {name: [] for name in factories}
    names = list(factories)
    post_fit = post_fit or {}

    for warmup in range(N_WARMUPS):
        order = names if warmup % 2 == 0 else list(reversed(names))
        for name in order:
            estimator = factories[name]()
            gc.collect()
            started = time.perf_counter()
            estimator.fit(X_train, y_train)
            if name in post_fit:
                post_fit[name](estimator)
            elapsed = time.perf_counter() - started
            del estimator
            print(f"warm-up {warmup + 1}/{N_WARMUPS}: {name} = {elapsed:.3f} s")

    for repeat in range(N_REPEATS):
        order = names if repeat % 2 == 0 else list(reversed(names))
        for name in order:
            estimator = factories[name]()
            gc.collect()
            started = time.perf_counter()
            estimator.fit(X_train, y_train)
            if name in post_fit:
                post_fit[name](estimator)
            elapsed = time.perf_counter() - started
            samples[name].append(elapsed)
            del estimator
            print(f"run {repeat + 1}/{N_REPEATS}: {name} = {elapsed:.3f} s")

    baseline_median = statistics.median(samples[baseline]) if baseline else None
    rows = []
    for name, times in samples.items():
        median = statistics.median(times)
        rows.append({
            "model_and_importance_workload": name,
            "median_elapsed_seconds": median,
            "min_elapsed_seconds": min(times),
            "max_elapsed_seconds": max(times),
            "speedup_vs_baseline": (
                baseline_median / median if baseline_median is not None else np.nan
            ),
        })
    summary = pd.DataFrame(rows).set_index("model_and_importance_workload")
    display(summary.round(3))
    if baseline:
        print("speedup_vs_baseline > 1 means this workload finished faster than the baseline.")
    return samples, summary

## 4. scikit-learn Random Forest vs. Bankai: permutation importance

The forests use the same tree count, Gini criterion, depth and leaf limits, feature sampling, bootstrap fraction, seed, and thread budget. Bankai is measured in both exact-split and 63-bin histogram modes, each with `importance_type="permutation"` and OOB accuracy decrease.

scikit-learn has no `importance_type="permutation"` option. Its row uses `sklearn.inspection.permutation_importance` with accuracy scoring on the held-out validation partition. The timed workload includes both `fit` and that importance calculation. To avoid nested parallelism, the fitted forest predicts with one worker while permutation importance parallelizes features across all available CPUs. Bankai's timer is its `fit` call, which includes its native OOB permutation calculation.

Both timed workloads include permutation importance, but the algorithms and evaluation protocols differ: scikit-learn uses five validation permutations per feature, while Bankai aggregates OOB permutations across trees.

In [ ]:
N_ESTIMATORS = 40
MAX_DEPTH = 20
MAX_LEAF_NODES = 511
MIN_SAMPLES_LEAF = 5
SAMPLE_FRACTION = 0.8
MODEL_SEED = 42
MAX_FEATURES = None  # use all features in all compared forests

COMMON_FOREST_PARAMS = dict(
    n_estimators=N_ESTIMATORS,
    criterion="gini",
    max_depth=MAX_DEPTH,
    max_leaf_nodes=MAX_LEAF_NODES,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    max_features=MAX_FEATURES,
    bootstrap=True,
    max_samples=SAMPLE_FRACTION,
    n_jobs=N_JOBS,
    random_state=MODEL_SEED,
)

MAX_BINS = 63
BIN_SAMPLE_SIZE = 200_000

In [ ]:
def make_sklearn_rf():
    return RandomForestClassifier(**COMMON_FOREST_PARAMS)


def make_bankai_exact(importance_type="permutation"):
    return BankaiRandomForestClassifier(
        **COMMON_FOREST_PARAMS,
        max_bins=None,
        importance_type=importance_type,
    )


def make_bankai_histogram(importance_type="gain"):
    return BankaiRandomForestClassifier(
        **COMMON_FOREST_PARAMS,
        max_bins=MAX_BINS,
        bin_sample_size=BIN_SAMPLE_SIZE,
        importance_type=importance_type,
    )

In [ ]:
PERMUTATION_REPEATS = 5
SKLEARN_PERMUTATION = "scikit-learn RF + validation permutation"


def calculate_sklearn_permutation(estimator):
    # Avoid nested parallelism: outer feature permutation uses all CPUs.
    estimator.set_params(n_jobs=1)
    return permutation_importance(
        estimator,
        X_validation,
        y_validation,
        scoring="accuracy",
        n_repeats=PERMUTATION_REPEATS,
        random_state=MODEL_SEED,
        n_jobs=N_JOBS,
    )


sklearn_factories = {
    SKLEARN_PERMUTATION: make_sklearn_rf,
    "Bankai exact + OOB permutation": lambda: make_bankai_exact("permutation"),
    "Bankai histogram (63 bins) + OOB permutation": lambda: make_bankai_histogram("permutation"),
}

sklearn_samples, sklearn_summary = run_fit_benchmark(
    sklearn_factories,
    baseline=SKLEARN_PERMUTATION,
    post_fit={SKLEARN_PERMUTATION: calculate_sklearn_permutation},
)

### Reading the scikit-learn comparison

The table reports total elapsed workload: scikit-learn fit plus validation permutation importance, versus Bankai fit with OOB permutation importance included. `speedup_vs_baseline > 1` means that Bankai completed its fit-and-importance workload faster than the scikit-learn reference. The calculation protocols differ as described above, so interpret the ratio as a workload comparison rather than an exact algorithm-to-algorithm equivalence.

## 5. LightGBM RF vs. Bankai: gain importance

This comparison remains **gain vs. gain** and fit-only. Both models use 40 trees/iterations, 63-bin histograms, depth 20, at most 511 leaves, at least 5 samples per leaf, all features, 80% row sampling, the same seed, and all available threads. Bankai's `importance_type="gain"` is paired with LightGBM's `importance_type="gain"`.

Bankai's `max_samples=0.8` with bootstrap is paired with LightGBM RF's `bagging_fraction=0.8` and `bagging_freq=1`. At the default 1,000 total rows, both fit on the same 800-row training partition, so all training rows are used to construct bins. LightGBM samples rows without replacement while Bankai bootstraps with replacement; their split criteria and tree-building internals also differ.

In [ ]:
LIGHTGBM_RF_PARAMS = dict(
    boosting_type="rf",
    objective="binary",
    n_estimators=N_ESTIMATORS,
    max_bin=MAX_BINS,
    max_depth=MAX_DEPTH,
    num_leaves=MAX_LEAF_NODES,
    min_child_samples=MIN_SAMPLES_LEAF,
    bagging_freq=1,
    bagging_fraction=SAMPLE_FRACTION,
    feature_fraction=1.0,
    subsample_for_bin=BIN_SAMPLE_SIZE,
    n_jobs=N_JOBS,
    random_state=MODEL_SEED,
    importance_type="gain",
    verbosity=-1,
)


def make_lightgbm_rf():
    return LGBMClassifier(**LIGHTGBM_RF_PARAMS)

lightgbm_factories = {
    "Bankai histogram (63 bins) + gain": lambda: make_bankai_histogram("gain"),
    "LightGBM RF + gain": make_lightgbm_rf,
}

lightgbm_samples, lightgbm_summary = run_fit_benchmark(
    lightgbm_factories,
    baseline="LightGBM RF + gain",
)

### Reading the LightGBM comparison

This table is fit-only with gain importance on both sides. Values above 1 in `speedup_vs_baseline` mean Bankai fit faster than LightGBM RF; values below 1 mean LightGBM fit faster. The comparison aligns major settings but does not make the algorithms or sampling distributions identical.